In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
import io

# 1. Carregando os dados (usando sua amostra)
dados = """codigo_ibge;cidade;ano;populacao;log_pop;dlog_pop;dlog_aluguel;caged_per_capita;selic_media;d_ic_br;dlog_incc_lag1;dlog_area
2304400;Fortaleza;2017;2436219.93;14.70;-0.0009;-0.0038;-0.0023;10.15;-0.0353;0.0594;1.2680
2304400;Fortaleza;2018;2433947.71;14.70;-0.0009;-0.0305;0.0023;6.58;0.1756;0.0416;-0.1164
2304400;Fortaleza;2019;2431677.61;14.70;-0.0009;0.0386;0.0005;6.03;0.0100;0.0376;1.2119
2304400;Fortaleza;2020;2429409.62;14.70;-0.0009;0.0148;0.0045;2.88;0.2149;0.0407;-0.0951
2304400;Fortaleza;2021;2427143.75;14.70;-0.0009;0.0368;0.0203;4.53;0.4996;0.0844;-0.4317
2304400;Fortaleza;2022;2424880.00;14.70;-0.0009;0.1997;0.0166;12.54;0.1840;0.1296;2.2338"""

df = pd.read_csv(io.StringIO(dados), sep=';')

# 2. Configurando o formato de Painel (MultiIndex: Entidade, Tempo)
# A linearmodels exige que o índice seja (cidade/id, ano)
df = df.set_index(['codigo_ibge', 'ano'])

# 3. Escolhendo variáveis e criando os Lags (0 e 1)
alvo = 'dlog_aluguel'
colunas_x = ['dlog_pop', 'caged_per_capita', 'selic_media', 'd_ic_br']

# Criando um DataFrame novo apenas com as colunas do modelo para organizar
df_modelo = df[[alvo]].copy()

for col in colunas_x:
    # Lag 0 (o ano atual)
    df_modelo[f'{col}_lag0'] = df[col]
    # Lag 1 (defasagem de 1 ano, respeitando o agrupamento da cidade no nível 0 do índice)
    df_modelo[f'{col}_lag1'] = df.groupby(level=0)[col].shift(1)

# A variável dlog_incc_lag1 já está defasada no seu banco, então entra direto
df_modelo['dlog_incc_lag1'] = df['dlog_incc_lag1']

# Remover as linhas com NaNs gerados pelo shift (perderemos o primeiro ano de cada cidade)
df_modelo = df_modelo.dropna()

# 4. Separando X e Y e adicionando a constante
y = df_modelo[alvo]
X = df_modelo.drop(columns=[alvo])
X = sm.add_constant(X) # Obrigatório para o modelo ter um intercepto basal

# 5. Rodando o Modelo de Efeitos Fixos
# entity_effects=True é o que diz ao modelo para criar um "efeito" único para cada cidade
modelo_fe = PanelOLS(y, X, entity_effects=True)
resultado = modelo_fe.fit(cov_type='robust') # Erros robustos são importantes em dados financeiros/demográficos

print(resultado.summary)

Total de combinações de lags a testar: 16

--- RESULTADO FINAL ---
Não foi possível encontrar um modelo viável com a quantidade de dados após o corte de NAs.
